# Clasificación Binaria de Tumores Mamarios



## Descenso de gradiente aplicado al Breast Cancer Wisconsin Dataset

## 1. Importación de librerías
Librerías necesarias y funciones creadas para cargar, preparar y probar el modelo base.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 2. Funciones de preparación del dataset


In [3]:
def descargar_cargar_datos():
    url = "https://raw.githubusercontent.com/selva86/datasets/master/BreastCancer.csv"
    df = pd.read_csv(url)

    df = df.drop(columns=['Id'], errors='ignore')
    df = df.dropna()

    valores_clase = df['Class'].unique()

    if not set(valores_clase).issubset({0, 1}):
        mapeo = {2: 0, 4: 1, '2': 0, '4': 1}
        df['Class'] = df['Class'].map(mapeo)

    df = df.dropna()
    return df


def normalizar_datos(df):
    y = df['Class'].values.reshape(-1, 1)
    X = df.drop(columns=['Class']).values

    x_min = X.min(axis=0)
    x_max = X.max(axis=0)

    X_normalizado = (X - x_min) / (x_max - x_min + 1e-8)
    return X_normalizado, y


def dividir_entrenamiento_prueba(X, y, porcentaje_entrenamiento=0.8):
    np.random.seed(42)
    indices = np.random.permutation(len(X))
    limite = int(len(X) * porcentaje_entrenamiento)

    indices_train = indices[:limite]
    indices_test = indices[limite:]

    X_train, X_test = X[indices_train], X[indices_test]
    y_train, y_test = y[indices_train], y[indices_test]

    return X_train, X_test, y_train, y_test

## 3. Modelo base

En esta sección se define la clase `NeuralNetwork`, que representa un modelo simple de clasificación binaria. El modelo calcula una combinación lineal de las variables de entrada y aplica una función sigmoide para obtener una probabilidad entre 0 y 1.

Esta parte corresponde al modelo base antes del entrenamiento.

In [4]:
class NeuralNetwork:
    def __init__(self, n_features):
        np.random.seed(42)

        self.W = np.random.rand(n_features, 1) * 0.01
        self.b = 0.0

    def sigmoid(self, z):
        return np.where(
            z >= 0,
            1 / (1 + np.exp(-z)),
            np.exp(z) / (1 + np.exp(z))
        )

    def forward(self, X):
        z = np.dot(X, self.W) + self.b
        y_hat = self.sigmoid(z)

        return y_hat

## 4. Carga y verificación inicial del dataset

Se carga el dataset y se revisan sus dimensiones para confirmar que los datos fueron importados correctamente.

In [5]:
datos = descargar_cargar_datos()

print("Dataset completo:", datos.shape)
datos.head()

Dataset completo: (683, 10)


,Cl.thickness,Cell.size,Cell.shape,Marg.adhesion,Epith.c.size,Bare.nuclei,Bl.cromatin,Normal.nucleoli,Mitoses,Class
0,5,1,1,1,2,1.0,3,1,1,0
1,5,4,4,5,7,10.0,3,2,1,0
2,3,1,1,1,2,2.0,3,1,1,0
3,6,8,8,1,3,4.0,3,7,1,0
4,4,1,1,3,2,1.0,3,1,1,0


## 5. Preparación de los datos

En esta sección se separan las características de entrada `X` y la variable objetivo `y`.

También se normalizan las características usando Min-Max Scaling para que todas las variables queden en un rango comparable. Esto es importante porque el descenso de gradiente puede comportarse mal si las variables tienen escalas muy diferentes.

In [6]:
X, y = normalizar_datos(datos)

print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)
print("Etiquetas encontradas en y:", set(y.flatten()))

Dimensiones de X: (683, 9)
Dimensiones de y: (683, 1)
Etiquetas encontradas en y: {np.int64(0), np.int64(1)}


La preparación de datos funcionó correctamente. El dataset quedó dividido en `X`, que contiene las 9 características utilizadas por el modelo, y `y`, que contiene la clase real de cada tumor.

La variable objetivo quedó codificada en formato binario, donde `0` representa tumor benigno y `1` representa tumor maligno.

## 6. División en entrenamiento y prueba

El dataset se divide en dos conjuntos:

- `train`: datos usados para entrenar el modelo.
- `test`: datos reservados para evaluar el desempeño final.

Se utiliza una semilla fija para que la división sea reproducible y el resultado sea el mismo cada vez que se ejecute el notebook.

In [7]:
X_train, X_test, y_train, y_test = dividir_entrenamiento_prueba(X, y)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (546, 9)
X_test: (137, 9)
y_train: (546, 1)
y_test: (137, 1)


La división de datos se realizó correctamente. El conjunto de entrenamiento quedó con 546 muestras y el conjunto de prueba con 137 muestras.

Cada muestra tiene 9 características, y cada etiqueta tiene una sola salida binaria. Esto confirma que los datos tienen dimensiones compatibles para alimentar el modelo de clasificación binaria.

## 7. Definición del modelo base

En esta sección se crea el modelo base de clasificación binaria.

La clase `NeuralNetwork` representa un modelo simple que recibe las características de entrada, calcula una combinación lineal y después aplica una función sigmoide para obtener una probabilidad entre 0 y 1.

En este punto el modelo todavía no está entrenado; solamente se verifica que pueda recibir los datos correctamente.

In [8]:
modelo = NeuralNetwork(n_features=X_train.shape[1])

print("Modelo creado correctamente.")
print("Número de características de entrada:", X_train.shape[1])
print("Dimensión de W:", modelo.W.shape)
print("Valor inicial de b:", modelo.b)

Modelo creado correctamente.
Número de características de entrada: 9
Dimensión de W: (9, 1)
Valor inicial de b: 0.0


## 8. Forward propagation inicial

Antes de entrenar el modelo, se realiza una primera propagación hacia adelante usando `X_train`.

El objetivo de esta prueba es comprobar que el modelo puede generar predicciones con dimensiones correctas. Como los pesos todavía no han sido ajustados mediante descenso de gradiente, se espera que las predicciones estén cerca de 0.5.

In [9]:
y_hat = modelo.forward(X_train)

print("Dimensiones de y_hat:", y_hat.shape)

print("\nPrimeras 5 predicciones:")
print(y_hat[:5])

print("\nRango de predicciones:")
print("mínimo:", y_hat.min())
print("máximo:", y_hat.max())

Dimensiones de y_hat: (546, 1)

Primeras 5 predicciones:
[[0.50418684]
 [0.50785804]
 [0.50007561]
 [0.50044523]
 [0.50001613]]

Rango de predicciones:
mínimo: 0.5
máximo: 0.5111345897654083
